In [1]:
import json
import numpy as np
import pandas as pd

# 1. Cargar matriz de transición A desde el JSON exportado
with open("../datos_procesados/resultados_hmm.json", "r", encoding="utf-8") as f:
    datos = json.load(f)

A = np.array(datos["matriz_transicion"])
estados = ["E0 (Reconnaissance)", "E1 (Initial Access)", "E2 (Exfiltration)", "E3 (Pivote)"]
acciones = ["d1: Monitoreo Pasivo", "d2: Inspección Profunda", "d3: Bloqueo Total"]

# 2. Definir la Matriz de Costos C (Filas: Acciones, Columnas: Estados futuros E0..E3)
matriz_costos = np.array([
    [0,   15, 100, 30],  # d1: Monitoreo Pasivo
    [5,    8,  45, 10],  # d2: Inspección Profunda
    [20,  12,   5,  8]   # d3: Bloqueo Total
])

df_costos = pd.DataFrame(matriz_costos, index=acciones, columns=estados)
print("--- Matriz de Costos Operativos C(Acción, Estado) ---")
print(df_costos)

# 3. Calcular Costo Esperado para cada estado actual Ei: CE = C * A^T
costos_esperados = matriz_costos @ A.T

df_resultados = pd.DataFrame(costos_esperados, index=acciones, columns=estados)
print("\n--- Costo Esperado por Acción dado el Estado Actual del HMM ---")
print(df_resultados.round(4))

# 4. Política Óptima de Decisión (mínimo costo esperado por columna)
print("\n--- Política Óptima de Defensa ---")
for col in df_resultados.columns:
    mejor_accion = df_resultados[col].idxmin()
    costo_min = df_resultados[col].min()
    print(f"Si el estado actual es {col:22s} -> {mejor_accion} (Costo Esperado: {costo_min:.2f})")

--- Matriz de Costos Operativos C(Acción, Estado) ---
                         E0 (Reconnaissance)  E1 (Initial Access)  \
d1: Monitoreo Pasivo                       0                   15   
d2: Inspección Profunda                    5                    8   
d3: Bloqueo Total                         20                   12   

                         E2 (Exfiltration)  E3 (Pivote)  
d1: Monitoreo Pasivo                   100           30  
d2: Inspección Profunda                 45           10  
d3: Bloqueo Total                        5            8  

--- Costo Esperado por Acción dado el Estado Actual del HMM ---
                         E0 (Reconnaissance)  E1 (Initial Access)  \
d1: Monitoreo Pasivo                  1.2039              15.8050   
d2: Inspección Profunda               5.3259               8.1563   
d3: Bloqueo Total                    19.6490              11.8572   

                         E2 (Exfiltration)  E3 (Pivote)  
d1: Monitoreo Pasivo               79